In [3]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    BatchNormalization,
    ReLU,
    Add,
    Dense
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Path
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"


# ============================================================
# 3. Download
# ============================================================

if not os.path.exists(DATA_PATH):

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(
            "/content/drive/MyDrive/Colab Notebooks"
        )

else:

    print("Existing dataset found")


# ============================================================
# 4. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128)
    # ->
    # (N, 128, 9)

    X = np.stack(X, axis=-1)

    y_path = (
        f"{DATA_PATH}/{split}/y_{split}.txt"
    )

    y = np.loadtxt(
        y_path
    ).astype(int)

    # 1~6 -> 0~5

    y = y - 1

    return X, y


X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# ============================================================
# 6. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 7. One-hot Encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 8. ResNet Block
# ============================================================

def residual_block(
    x,
    filters,
    kernel_size=3
):

    shortcut = x

    # Main path

    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)

    x = ReLU()(x)

    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)


    # Shortcut

    if shortcut.shape[-1] != filters:

        shortcut = Conv1D(
            filters,
            1,
            padding="same",
            use_bias=False
        )(shortcut)

        shortcut = BatchNormalization()(shortcut)


    # Skip Connection

    x = Add()([
        x,
        shortcut
    ])

    x = ReLU()(x)

    return x


# ============================================================
# 9. ResNet Model
# ============================================================

inputs = Input(
    shape=(128, 9)
)


x = Conv1D(
    32,
    3,
    padding="same",
    use_bias=False
)(inputs)

x = BatchNormalization()(x)
x = ReLU()(x)


x = residual_block(
    x,
    32
)

x = MaxPooling1D(
    2
)(x)


x = residual_block(
    x,
    64
)

x = MaxPooling1D(
    2
)(x)


x = residual_block(
    x,
    128
)


x = GlobalAveragePooling1D()(x)


x = Dense(
    128,
    activation="relu"
)(x)


outputs = Dense(
    6,
    activation="softmax"
)(x)


model = Model(
    inputs,
    outputs
)


# ============================================================
# 10. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 11. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 12. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 13. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("RESNET TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 14. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 15. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")

print(cm)

Existing dataset found
X_train: (7352, 128, 9)
X_test : (2947, 128, 9)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 128, 32)   │        864 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 32)   │        128 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 32)   │          0 │ batch_normalizat… │
│                     │                   │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 128, 32)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 64, 32)    │          0 │ re_lu_2[0][0]     │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 64, 64)    │      6,144 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_7[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 64, 64)    │     12,288 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 64, 64)    │      2,048 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat

 Total params: 129,382 (505.40 KB)

 Trainable params: 128,038 (500.15 KB)

 Non-trainable params: 1,344 (5.25 KB)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 95ms/step - accuracy: 0.9097 - loss: 0.2563 - val_accuracy: 0.9014 - val_loss: 0.3500
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 114ms/step - accuracy: 0.9510 - loss: 0.1160 - val_accuracy: 0.9055 - val_loss: 0.2881
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9563 - loss: 0.0975 - val_accuracy: 0.8892 - val_loss: 0.3520
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 110ms/step - accuracy: 0.9606 - loss: 0.0895 - val_accuracy: 0.8912 - val_loss: 0.4500
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 113ms/step - accuracy: 0.9617 - loss: 0.0847 - val_accuracy: 0.8919 - val_loss: 0.5137
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accuracy: 0.9641 - loss: 0.0805 - val_accuracy: 0.9014 - val_loss: 0.5548
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.9629 - loss: 0.0826 - val_accuracy: 0.9035 - val_loss: 0.4988
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 107ms/step - accuracy: 0.9624 - loss: 0.0757 - val_accura